In [1]:
cd ..

/home/karaman/Documents/bet


In [16]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.churn_label_generator import generate_churn_labels



In [17]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")


In [19]:
df_churn = generate_churn_labels(sessions_silver, config_)
df_churn = (
    df_churn
    .join(
        players_silver.select("player_id"),
        on="player_id",
        how="inner"
    )
)
#df_churn.write.mode("append").parquet("./data/bronze/churn_labels")


In [20]:
players_silver.show()

+---------+----------+-----------------+-------+----------+-----------+-------------------+---------------+------------+------------------+
|player_id|player_idx|registration_date|country|age_bucket|device_type|acquisition_channel|lifecycle_stage|risk_segment|   current_balance|
+---------+----------+-----------------+-------+----------+-----------+-------------------+---------------+------------+------------------+
|     P100|       100|       2024-04-19|     GR|     25-34|    desktop|            organic|        engaged|         low|            276.19|
|   P10000|     10000|       2024-04-14|     EN|     25-34|     mobile|            organic|        engaged|         low|224.10999999999999|
|   P10005|     10005|       2024-04-03|     GR|     35-44|     mobile|               paid|        engaged|         low|            211.91|
|   P10006|     10006|       2024-03-05|     GR|     35-44|    desktop|          affiliate|        engaged|         low|            201.89|
|   P10010|     1001

In [23]:
player_snapshot = (players_silver
                   .select('player_id','player_idx','country','age_bucket','device_type','acquisition_channel', 'registration_date', 'risk_segment', 'lifecycle_stage', F.col('current_balance').alias('current_balance'))
                   .join(df_churn.select('player_id','last_session_date'),
                         on='player_id',
                         how='left')
                         .withColumn('days_since_last_session', 
                                     F.date_diff(F.lit(config_.end_date), F.col('last_session_date')))
                         .drop('player_id', 'reference_date')

)
player_snapshot.show()

+----------+-------+----------+-----------+-------------------+-----------------+------------+---------------+------------------+-----------------+-----------------------+
|player_idx|country|age_bucket|device_type|acquisition_channel|registration_date|risk_segment|lifecycle_stage|   current_balance|last_session_date|days_since_last_session|
+----------+-------+----------+-----------+-------------------+-----------------+------------+---------------+------------------+-----------------+-----------------------+
|      1001|     EN|     18-24|    desktop|          affiliate|       2024-04-01|         low|        engaged|156.82000000000002|       2024-06-30|                      0|
|     10013|     GR|     35-44|    desktop|               paid|       2024-02-26|     unknown|            new|             93.47|             NULL|                   NULL|
|     10014|     EN|     18-24|    desktop|            organic|       2024-03-23|      medium|        engaged|            198.61|       2024

## Window functions for rolling computations 7days or 30days ago

In [ ]:

sessions_silver_sec = (sessions_silver
                   .withColumn('session_date_days', 
                F.datediff(F.col("session_date"), F.lit("1970-01-01"))
)
)

window_7d = (Window.partitionBy('player_id').orderBy('session_date_days').rangeBetween(-7,0))
window_30d = (Window.partitionBy('player_id').orderBy('session_date_days').rangeBetween(-30,0))
sessions_silver_sec = (sessions_silver_sec
            .withColumn('num_sessions_7d', F.count('session_id').over(window_7d))
            .withColumn('num_sessions_30d', F.count('session_id').over(window_30d))
            .withColumn('avg_sessions_duration_30d', F.avg('session_duration_sec').over(window_30d))
)

((sessions_silver_sec.groupBy('player_id')
 .agg(
     F.max('session_date_days').alias('session_date_days'))
)
.join(sessions_silver_sec.select('player_id', 'session_date_days','session_date','num_sessions_7d','num_sessions_30d','avg_sessions_duration_30d'), on=['player_id', 'session_date_days'], how='inner')
.show()
)



## player_behavior

In [38]:
player_behavior = (sessions_silver.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('session_date')))
 .filter(F.col('days_diff') <= 30)
 .groupBy('player_id')
 .agg(
            F.sum(F.when(F.col('days_diff') <= 7, 1).otherwise(0)).alias('sessions_7d'),
            F.sum(F.when(F.col('days_diff') <= 7, F.col('signed_amount')).otherwise(0)).alias('net_game_result_7d'),

            F.count('*').alias('sesions_30d'),
            F.avg(F.col('session_duration_sec')).cast('int').alias('avg_session_duration_sec_30d'),
            F.avg(F.col('total_bet_amount')).alias('avg_bet_amount_30d'),
            F.sum(F.col('signed_amount')).alias('net_game_result_30d'),
 )
                
)
player_behavior.show()

+---------+-----------+------------------+-----------+----------------------------+------------------+-------------------+
|player_id|sessions_7d|net_game_result_7d|sesions_30d|avg_session_duration_sec_30d|avg_bet_amount_30d|net_game_result_30d|
+---------+-----------+------------------+-----------+----------------------------+------------------+-------------------+
|   P42501|          6|            341.76|         17|                        1991| 50.53058823529411|              956.9|
|   P46900|          4|            253.59|         11|                        1793| 55.50545454545454|  851.3499999999999|
|   P51902|          4|            129.56|         14|                        1733| 43.11714285714286|             785.03|
|   P89172|          2|            158.51|         12|                        1271| 57.98916666666667|              674.9|
|   P49124|          6|            296.94|         19|                        1588| 55.28842105263158|            1184.49|
|   P29844|     

#### All money events

In [25]:

silver_money_events = ( sessions_silver.withColumn('event_type', F.lit('session'))
                .select('player_id', 
                    F.col('session_id').alias('event_id'),   
                    F.col('session_date').alias('event_ts'), 
                    'event_type',
                'signed_amount',
                'balance_after_txn')
                .unionByName(transactions_silver
                .select('player_id', 
                  F.col('transaction_id').alias('event_id'),   
                    F.col('transaction_ts').alias('event_ts'),
                     F.col('transaction_type').alias('event_type'),
                    'signed_amount',
                    'balance_after_txn'))

)
silver_money_events.show()

+---------+--------------------+-------------------+----------+-------------+------------------+
|player_id|            event_id|           event_ts|event_type|signed_amount| balance_after_txn|
+---------+--------------------+-------------------+----------+-------------+------------------+
|   P59929|0000068b-b799-460...|2024-04-11 00:00:00|   session|        28.08| 775.0700000000002|
|   P17854|0000550f-ac54-42d...|2024-05-25 00:00:00|   session|        92.12|           2585.93|
|   P14053|0000645d-4467-408...|2024-05-18 00:00:00|   session|        37.12|            1545.7|
|   P23612|000097ec-eae7-4a8...|2024-02-18 00:00:00|   session|        80.69| 789.2399999999998|
|   P72737|0000b0f6-ebb6-450...|2024-06-29 00:00:00|   session|        33.74| 4535.270000000001|
|   P29031|0000b85c-f5c0-4ee...|2024-02-26 00:00:00|   session|        97.19|1631.6999999999998|
|    P5854|0000c4d1-617e-479...|2024-06-26 00:00:00|   session|        61.15|           2387.34|
|   P59616|00010a87-29c2-44d..

In [ ]:

### Correct ###
player_behavior = (silver_money_events.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('event_ts')))
 .filter(F.col('days_diff') <= 30)
 .groupBy('player_id')
 .agg(
            F.sum(F.when(F.col('days_diff') <= 7, F.col('signed_amount')).otherwise(0)).alias('net_amount_result_7d'),
            F.sum(F.col('signed_amount')).alias('net_amount_result_30d'),
 )        
)
player_behavior.show()

+---------+--------------------+---------------------+
|player_id|net_amount_result_7d|net_amount_result_30d|
+---------+--------------------+---------------------+
|   P42501|              341.76|                956.9|
|   P46900|              253.59|    851.3499999999999|
|   P51902|              129.56|               785.03|
|   P89172|              158.51|                674.9|
|   P49124|              268.34|              1155.89|
|   P29844|  238.33999999999997|               935.88|
|   P97745|  107.70000000000002|    929.2700000000001|
|   P87052|               52.67|               550.68|
|   P74847|              254.06|    953.9499999999998|
|   P87893|              104.69|   477.24000000000007|
|   P75373|              340.18|              1257.56|
|   P96483|               19.64|    850.1100000000001|
|   P14522|              136.06|    858.1300000000001|
|   P31528|              297.01|               777.24|
|   P96398|               199.3|   1238.6399999999999|
|   P51812

In [33]:
w = Window.partitionBy("player_id").orderBy(F.col("event_ts").desc())
player_30d = (silver_money_events.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('event_ts')))
 .filter(F.col('days_diff') >=30)
 .withColumn('rn', F.row_number().over(w))
.filter(F.col('rn') ==1)
)
player_30d_act= (players_silver
                   .select('player_id','player_idx','current_balance')
                   .join(player_30d.select('player_id',F.col('balance_after_txn')),
                         on='player_id',
                         how='left')
                             .withColumn(
                              "balance_30d_ago",
                              F.coalesce("balance_after_txn", "current_balance"))
                              .drop('player_id', 'current_balance', 'balance_after_txn')
)
player_30d_act.show()

+----------+------------------+
|player_idx|   balance_30d_ago|
+----------+------------------+
|      1001|1546.8999999999999|
|     10013|             93.47|
|     10014|1749.2900000000002|
|     10022|            107.63|
|     10027|           5140.32|
|     10033|1150.3700000000001|
|     10034|           4414.48|
|     10039|           1072.66|
|     10042| 2961.500000000001|
|     10044|1444.3500000000001|
|     10049|3586.3800000000015|
|     10057|189.10999999999999|
|     10002|             62.99|
|     10008|             112.3|
|      1002| 3705.259999999999|
|     10021| 4232.390000000001|
|     10025|            196.49|
|     10032|2727.0800000000004|
|     10041|              7.12|
|     10043| 3589.890000000001|
+----------+------------------+
only showing top 20 rows


In [34]:
w = Window.partitionBy("player_id").orderBy(F.col("event_ts").desc())
player_7d = (silver_money_events.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('event_ts')))
 .filter(F.col('days_diff') >=7)
 .withColumn('rn', F.row_number().over(w))
.filter(F.col('rn') ==1)
)
player_7d_act= (players_silver
                   .select('player_id','player_idx','current_balance')
                   .join(player_7d.select('player_id',F.col('balance_after_txn')),
                         on='player_id',
                         how='left')
                             .withColumn(
                              "balance_7d_ago",
                              F.coalesce("balance_after_txn", "current_balance"))
                              .drop('player_id', 'current_balance', 'balance_after_txn')
)
player_7d_act.show()

+----------+------------------+
|player_idx|    balance_7d_ago|
+----------+------------------+
|      1001|2193.7499999999995|
|     10013|             93.47|
|     10014|2603.1000000000004|
|     10022|            107.63|
|     10027| 5981.420000000001|
|     10033|1724.6000000000001|
|     10034|           4928.18|
|     10039|           1172.68|
|     10042|3707.1500000000015|
|     10044|2272.3900000000003|
|     10049| 4210.980000000001|
|     10057|189.10999999999999|
|     10002|             62.99|
|     10008|             112.3|
|      1002| 4163.519999999999|
|     10021| 4813.550000000001|
|     10025|            196.49|
|     10032|3585.5099999999998|
|     10041|              7.12|
|     10043| 4108.160000000001|
+----------+------------------+
only showing top 20 rows


In [36]:
silver_money_events.filter(F.col('player_id')=='P10022').show()

+---------+--------+--------+----------+-------------+-----------------+
|player_id|event_id|event_ts|event_type|signed_amount|balance_after_txn|
+---------+--------+--------+----------+-------------+-----------------+
+---------+--------+--------+----------+-------------+-----------------+



In [37]:
players_silver.filter(F.col('player_id')=='P10013').show()

+---------+----------+-----------------+-------+----------+-----------+-------------------+---------------+------------+---------------+
|player_id|player_idx|registration_date|country|age_bucket|device_type|acquisition_channel|lifecycle_stage|risk_segment|current_balance|
+---------+----------+-----------------+-------+----------+-----------+-------------------+---------------+------------+---------------+
|   P10013|     10013|       2024-02-26|     GR|     35-44|    desktop|               paid|            new|     unknown|          93.47|
+---------+----------+-----------------+-------+----------+-----------+-------------------+---------------+------------+---------------+

